In [107]:
!pip install peft accelerate rouge-score -q


In [108]:
import bitsandbytes
print(f"Version: {bitsandbytes.__version__}")
print(f"Location: {bitsandbytes.__file__}")

import torch
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

import bitsandbytes as bnb
print(dir(bnb))
print(bnb.cuda_specs)

Version: 0.49.2
Location: /usr/local/lib/python3.12/dist-packages/bitsandbytes/__init__.py
CUDA available: True
GPU: NVIDIA A100-SXM4-40GB
['MatmulLtState', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__pdoc__', '__spec__', '__version__', '_import_backends', '_ops', 'adam', 'autograd', 'backends', 'cextension', 'consts', 'cpu_ops', 'cuda_ops', 'cuda_specs', 'default_ops', 'features', 'functional', 'importlib', 'matmul', 'matmul_4bit', 'modules', 'nn', 'optim', 'research', 'supported_torch_devices', 'sys', 'torch', 'triton', 'utils']
<module 'bitsandbytes.cuda_specs' from '/usr/local/lib/python3.12/dist-packages/bitsandbytes/cuda_specs.py'>


In [ ]:
# Mount Google Drive to access data
from google.colab import drive
drive.mount('/content/drive')

import json
import random
import itertools
from collections import defaultdict
from PIL import Image

PROJECT_DIR = '/content/drive/MyDrive/derma_project'
DATA_DIR = f'{PROJECT_DIR}/data'
JSON_PATH = f'{DATA_DIR}/generated_reports_clean.json'

with open(JSON_PATH, 'r') as f:
    reports = json.load(f)

print(f"Loaded {len(reports)} reports")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loaded 2376 reports


In [ ]:
# Shuffle and split the dataset into training and validation sets
random.seed(42)
shuffled = reports.copy()
random.shuffle(shuffled)

split_idx = int(len(shuffled) * 0.9)    # 90% for training, 10% for validation
train_reports = shuffled[:split_idx]
val_reports = shuffled[split_idx:]

print(f"Train: {len(train_reports)}")
print(f"Val: {len(val_reports)}")

Train: 2138
Val: 238


In [ ]:
# Group validation reports by diagnosis for balanced sampling (stratified sampling)
val_reports_by_dx = defaultdict(list)
for r in val_reports:
    val_reports_by_dx[r['diagnosis']].append(r)

eval_sample = []

# Sample up to 13 examples per diagnosis for evaluation
for dx, examples in val_reports_by_dx.items():
    n = min(13, len(examples))
    sampled = random.sample(examples, n)
    eval_sample.append(sampled)
    print(f"{dx}: {n} examples sampled")

eval_sample = list(itertools.chain.from_iterable(eval_sample))
print(f"\nTotal evaluation examples: {len(eval_sample)}")

bkl: 13 examples sampled
bcc: 13 examples sampled
nv: 13 examples sampled
mel: 13 examples sampled
akiec: 13 examples sampled
vasc: 13 examples sampled
df: 13 examples sampled

Total evaluation examples: 91


In [ ]:
# Test ROUGE-L score between two sample sentences
# ROUGE-L measure is used to evaluate the quality of generated reports against reference reports
# Score ranges from 0 to 1, with higher scores indicating better overlap between generated and reference reports
!pip install rouge-score -q
from rouge_score import rouge_scorer

scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)

score = scorer.score(
    "the lesion has irregular borders",
    "the lesion displays irregular and asymmetric borders"
)
print(f"Test ROUGE-L score: {score['rougeL'].fmeasure:.3f}")

Test ROUGE-L score: 0.667


In [ ]:
import torch

# Generate a report with metadata (age, sex, localization)
def generate_report_with_metadata(model, processor, image_path, age, sex, localization):
    image = Image.open(image_path).convert("RGB")

    prompt = f"""<|begin_of_text|><|start_header_id|>user<|end_header_id|>

<|image|>You are a dermatology assistant analyzing a dermatoscopic image.

Patient information:
- Age: {age} years old
- Sex: {sex}
- Lesion location: {localization}

Analyze the visual features of this lesion and produce a structured clinical report.<|eot_id|><|start_header_id|>assistant<|end_header_id|>

"""

    image_inputs = processor.image_processor(image, return_tensors="pt")
    text_inputs = processor.tokenizer(prompt, return_tensors="pt")

    inputs = {
        'input_ids': text_inputs['input_ids'].to(model.device),
        'attention_mask': text_inputs['attention_mask'].to(model.device),
        'pixel_values': image_inputs['pixel_values'].to(model.device),
    }

    if 'aspect_ratio_ids' in image_inputs:
        inputs['aspect_ratio_ids'] = image_inputs['aspect_ratio_ids'].to(model.device)
    if 'aspect_ratio_mask' in image_inputs:
        inputs['aspect_ratio_mask'] = image_inputs['aspect_ratio_mask'].to(model.device)

    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=300)

    return processor.tokenizer.decode(output[0], skip_special_tokens=True)

# Generate a report without metadata
def generate_report_without_metadata(model, processor, image_path):
    image = Image.open(image_path).convert("RGB")

    prompt = """<|begin_of_text|><|start_header_id|>user<|end_header_id|>

<|image|>You are a dermatology assistant. Analyze the visual features of this lesion and produce a structured clinical report.<|eot_id|><|start_header_id|>assistant<|end_header_id|>

"""

    image_inputs = processor.image_processor(image, return_tensors="pt")
    text_inputs = processor.tokenizer(prompt, return_tensors="pt")

    inputs = {
        'input_ids': text_inputs['input_ids'].to(model.device),
        'attention_mask': text_inputs['attention_mask'].to(model.device),
        'pixel_values': image_inputs['pixel_values'].to(model.device),
    }

    if 'aspect_ratio_ids' in image_inputs:
        inputs['aspect_ratio_ids'] = image_inputs['aspect_ratio_ids'].to(model.device)
    if 'aspect_ratio_mask' in image_inputs:
        inputs['aspect_ratio_mask'] = image_inputs['aspect_ratio_mask'].to(model.device)

    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=300)

    return processor.tokenizer.decode(output[0], skip_special_tokens=True)

print("Functions updated")

Functions updated


In [ ]:
# Run ROUGE evaluation on the sampled validation set
def run_rouge_evaluation(model, processor, eval_sample, scorer):
    results = []
    
    for i, item in enumerate(eval_sample):
        print(f"Processing {i+1}/{len(eval_sample)}: {item['image_id']} ({item['diagnosis']})")

        reference = item['report']  # The reference report from the dataset

        # Generate reports with and without metadata
        report_with = generate_report_with_metadata(
            model, processor,
            item['image_path'],
            item['age'],
            item['sex'],
            item['localization']
        )

        report_without = generate_report_without_metadata(
            model, processor,
            item['image_path']
        )

        # Compute ROUGE-L scores against the reference report
        score_with = scorer.score(reference, report_with)['rougeL'].fmeasure
        score_without = scorer.score(reference, report_without)['rougeL'].fmeasure

        # Store results for analysis
        results.append({
            'image_id':       item['image_id'],
            'diagnosis':      item['diagnosis'],
            'score_with':     score_with,
            'score_without':  score_without,
            'report_with':    report_with,
            'report_without': report_without,
            'reference':      reference
        })

    return results

In [ ]:
def analyze_results(results):
    import numpy as np

    # Extract scores for both conditions
    scores_with = [r['score_with'] for r in results]
    scores_without = [r['score_without'] for r in results]

    # Print overall statistics (mean, std, min, max)
    print("=" * 50)
    print("ROUGE-L EVALUATION RESULTS")
    print("=" * 50)
    print(f"\nCondition A (image only):")
    print(f"  Mean ROUGE-L:   {np.mean(scores_without):.4f}")
    print(f"  Std:            {np.std(scores_without):.4f}")
    print(f"  Min:            {np.min(scores_without):.4f}")
    print(f"  Max:            {np.max(scores_without):.4f}")

    print(f"\nCondition B (image + metadata):")
    print(f"  Mean ROUGE-L:   {np.mean(scores_with):.4f}")
    print(f"  Std:            {np.std(scores_with):.4f}")
    print(f"  Min:            {np.min(scores_with):.4f}")
    print(f"  Max:            {np.max(scores_with):.4f}")

    # Calculate the average improvement from metadata
    improvement = np.mean(scores_with) - np.mean(scores_without)
    print(f"\nImprovement from metadata: {improvement:+.4f}")

    if improvement > 0:
        print("Hypothesis SUPPORTED — metadata improves report quality")
    else:
        print("Hypothesis NOT SUPPORTED — metadata did not improve scores")

    # Break down results by diagnosis category to see if certain conditions benefit more from metadata
    print("\nResults by diagnosis category:")
    by_dx = {}
    for r in results:
        dx = r['diagnosis']
        if dx not in by_dx:
            by_dx[dx] = {'with': [], 'without': []}
        by_dx[dx]['with'].append(r['score_with'])
        by_dx[dx]['without'].append(r['score_without'])

    # Print average scores by diagnosis
    for dx, scores in sorted(by_dx.items()):
        avg_with = np.mean(scores['with'])
        avg_without = np.mean(scores['without'])
        diff = avg_with - avg_without
        print(f"  {dx:6s}: with={avg_with:.3f}  without={avg_without:.3f}  diff={diff:+.3f}")

print("Analysis function defined")

Analysis function defined


In [ ]:
# Save results to a JSON file for further analysis or visualization
def save_results(results):
    RESULTS_PATH = f'{PROJECT_DIR}/data/rouge_results.json'
    with open(RESULTS_PATH, 'w') as f:
        json.dump(results, f)
    print(f"Results saved to {RESULTS_PATH}")

In [ ]:
# Load Llama-3.2-11B-Vision-Instruct model and processor, then load the fine-tuned model for evaluation
from peft import PeftModel
from transformers import MllamaForConditionalGeneration, AutoProcessor
import torch

SAVE_PATH = f'{PROJECT_DIR}/checkpoints/final_model'
MODEL_ID = "meta-llama/Llama-3.2-11B-Vision-Instruct"

base_model = MllamaForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    use_fast=False
)

model = PeftModel.from_pretrained(base_model, SAVE_PATH)
model.eval()

print("Fine-tuned model loaded successfully")
print(f"Memory allocated: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/906 [00:00<?, ?it/s]

Fine-tuned model loaded successfully
Memory allocated: 40.4 GB


In [ ]:
# Full evaluation pipeline: run ROUGE evaluation, analyze results, and save results to a file
results = run_rouge_evaluation(model, processor, eval_sample, scorer)
analyze_results(results)
save_results(results)

Processing 1/91: ISIC_0025388 (bkl)
Processing 2/91: ISIC_0033676 (bkl)
Processing 3/91: ISIC_0031584 (bkl)
Processing 4/91: ISIC_0027454 (bkl)
Processing 5/91: ISIC_0033900 (bkl)
Processing 6/91: ISIC_0028611 (bkl)
Processing 7/91: ISIC_0026301 (bkl)
Processing 8/91: ISIC_0026416 (bkl)
Processing 9/91: ISIC_0033507 (bkl)
Processing 10/91: ISIC_0033587 (bkl)
Processing 11/91: ISIC_0033262 (bkl)
Processing 12/91: ISIC_0031329 (bkl)
Processing 13/91: ISIC_0024435 (bkl)
Processing 14/91: ISIC_0028671 (bcc)
Processing 15/91: ISIC_0026350 (bcc)
Processing 16/91: ISIC_0024897 (bcc)
Processing 17/91: ISIC_0025031 (bcc)
Processing 18/91: ISIC_0033146 (bcc)
Processing 19/91: ISIC_0026855 (bcc)
Processing 20/91: ISIC_0026643 (bcc)
Processing 21/91: ISIC_0032497 (bcc)
Processing 22/91: ISIC_0031728 (bcc)
Processing 23/91: ISIC_0028693 (bcc)
Processing 24/91: ISIC_0029820 (bcc)
Processing 25/91: ISIC_0033205 (bcc)
Processing 26/91: ISIC_0030755 (bcc)
Processing 27/91: ISIC_0029861 (nv)
Processing 

In [ ]:
# Helper function to extract diagnosis from generated reports for accuracy evaluation
def extract_diagnosis(report_text):
    report_lower = report_text.lower()

    diagnosis_map = {
        'melanoma':              'mel',
        'melanocytic nevus':     'nv',
        'nevus':                 'nv',
        'basal cell carcinoma':  'bcc',
        'actinic keratosis':     'akiec',
        'benign keratosis':      'bkl',
        'keratosis':             'bkl',
        'vascular lesion':       'vasc',
        'dermatofibroma':        'df',
    }

    for term, code in diagnosis_map.items():
        if term in report_lower:
            return code

    return 'unknown'

# Evaluate diagnosis accuracy by comparing extracted diagnoses from generated reports to the true diagnosis in the dataset
def evaluate_diagnosis_accuracy(results):
    correct_with = 0
    correct_without = 0
    unknown_with = 0
    unknown_without = 0
    total = len(results)

    per_category = {}

    # Check if predicted diagnosis matches true diagnosis for both conditions, and count unknowns
    for r in results:
        true_dx = r['diagnosis']
        pred_with = extract_diagnosis(r['report_with'])
        pred_without = extract_diagnosis(r['report_without'])

        if true_dx not in per_category:
            per_category[true_dx] = {
                'correct_with': 0,
                'correct_without': 0,
                'total': 0
            }

        per_category[true_dx]['total'] += 1

        if pred_with == true_dx:
            correct_with += 1
            per_category[true_dx]['correct_with'] += 1
        if pred_with == 'unknown':
            unknown_with += 1

        if pred_without == true_dx:
            correct_without += 1
            per_category[true_dx]['correct_without'] += 1
        if pred_without == 'unknown':
            unknown_without += 1

    # Print overall accuracy results and breakdown by diagnosis category
    print("=" * 50)
    print("DIAGNOSIS ACCURACY RESULTS")
    print("=" * 50)
    print(f"\nCondition A (image only):")
    print(f"  Accuracy:  {correct_without/total:.3f} ({correct_without}/{total})")
    print(f"  Unknown:   {unknown_without}")

    print(f"\nCondition B (image + metadata):")
    print(f"  Accuracy:  {correct_with/total:.3f} ({correct_with}/{total})")
    print(f"  Unknown:   {unknown_with}")

    print(f"\nPer category breakdown:")
    for dx, counts in sorted(per_category.items()):
        acc_with = counts['correct_with'] / counts['total']
        acc_without = counts['correct_without'] / counts['total']
        print(f"  {dx:6s}: with={acc_with:.2f}  without={acc_without:.2f}  n={counts['total']}")

print("Functions defined")

Functions defined


In [121]:
evaluate_diagnosis_accuracy(results)


DIAGNOSIS ACCURACY RESULTS

Condition A (image only):
  Accuracy:  0.242 (22/91)
  Unknown:   8

Condition B (image + metadata):
  Accuracy:  0.209 (19/91)
  Unknown:   5

Per category breakdown:
  akiec : with=0.00  without=0.00  n=13
  bcc   : with=0.15  without=0.00  n=13
  bkl   : with=0.15  without=0.23  n=13
  df    : with=0.15  without=0.31  n=13
  mel   : with=0.08  without=0.23  n=13
  nv    : with=0.69  without=0.69  n=13
  vasc  : with=0.23  without=0.23  n=13
